# Trending commodity news: what is moving gold, silver, copper and oil right now?

**Powered by [Bigdata.com](https://bigdata.com)** · [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bigdata-com/bigdata-trending-news/blob/main/notebooks/02_commodities_trending.ipynb)

**The question:** *"Show me trending commodity headlines, such as gold or silver price up or down, where there is
significant interest and engagement around similar articles."*

This notebook finds the commodity stories that many outlets are covering **right now**, tells you whether each one
points **up or down** for the commodity's price, and flags early, strongly-worded stories that have not spread yet.

### What "trending" means here

Bigdata.com does **not** capture reader engagement (clicks, reads, shares). Trending is measured from
**publishing momentum**, which is a strong proxy for attention:

| Signal | Question it answers | Measured from |
|---|---|---|
| **Breadth** | How many articles and *distinct outlets* are covering the **same story**? (the "engagement around similar articles") | articles grouped into stories |
| **Acceleration** | Is this entity covered more than usual right now? (`spike = now ÷ normal`) | `/v1/search/volume` |
| **Recency** | How fresh is the latest coverage? | article timestamps |

A story needs **at least 2 independent outlets** to count as trending. Stories with strong positive or negative
sentiment from credible outlets that have *not yet* built wide coverage are shown separately as **Emerging**.

> If you have your own click or read data, it can be blended in later through the
> `external_engagement_score` hook in `trending/rank.py` (default 0).

### What this notebook does

| Step | What happens | Uses an AI model? |
|---|---|---|
| 1. Resolve | Commodity names → Bigdata.com entity IDs (Knowledge Graph); keyword mode if not found | No |
| 2. Momentum | Is the commodity covered more than usual right now? | No |
| 3. Fetch | Pull the current window's news for each commodity | No |
| 4. Relevance gate | Keep articles about the **market** (price, supply, demand, flows); drop miner promos and noise | Yes (falls back to rules) |
| 5. Group | Group articles that report the **same event** into one story | Yes (falls back to text similarity) |
| 6. Enrich | Title, summary, **direction (up / down / mixed)**, driver, materiality | Yes (falls back to keyword rules) |
| 7. Rank | Trend score per commodity, plus an **Emerging** list per commodity | No |
| 8. Report | HTML report grouped by commodity + CSV files in `outputs/` | No |

All counts, momentum figures and sentiment values are computed from Bigdata.com data. The AI model only filters,
groups, titles and labels articles. Run the cells top to bottom (about 1-2 minutes). Each section shows its typical duration, and every code cell prints how long it took (`⏱`).

### Running in Google Colab

Click the badge above, then run the cells top to bottom. The next cell prepares Colab (it does nothing anywhere else):
it downloads the code, installs the packages and reads your API keys from Colab **Secrets** (🔑 icon in the left
sidebar). Add `BIGDATA_API_KEY` and `OPENAI_API_KEY` there
and switch on *Notebook access*. A `BIGDATA_API_KEY` is required.
While the repository is private, also add a `GITHUB_TOKEN` secret (a GitHub token with read access to it).

In [23]:
# 0. Google Colab only: download the code, install packages, load API keys from Colab Secrets
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    from google.colab import userdata

    def secret(name):
        try:
            return userdata.get(name)
        except Exception:  # secret missing or notebook access not granted
            return None

    repo, target = "Bigdata-com/bigdata-trending-news", "/content/bigdata-trending-news"
    if not os.path.exists(target):
        token = secret("GITHUB_TOKEN")   # only needed while the repository is private
        auth = f"x-access-token:{token}@" if token else ""
        subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{repo}.git", target], check=True)
        subprocess.run(["git", "-C", target, "remote", "set-url", "origin", f"https://github.com/{repo}.git"])
    # Colab already has pandas and Jupyter; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "python-dotenv", "jinja2",
                    "openai"], check=True)
    os.chdir(f"{target}/notebooks")
    for name in ["BIGDATA_API_KEY", "OPENAI_API_KEY", "LLM_MODEL"]:
        if value := secret(name):
            os.environ[name] = value
    print("Colab ready. Keys found:", [n for n in ["BIGDATA_API_KEY", "OPENAI_API_KEY"]
                                      if os.environ.get(n)] or "none")
else:
    print("Not running in Colab: nothing to do.")

Not running in Colab: nothing to do.
⏱ 0.0 s


In [24]:
# Setup: make the shared `trending` package importable, tidy pandas output, time every cell.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from trending.timing import cell_timer
cell_timer.start()   # from here on, every cell ends with "⏱ <seconds>"; the last cell shows a summary

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from trending import config as cfg
from trending.pipeline import start, build_stories, finish, llm_label
from trending.resolve import resolve_all, entity_table
from trending.momentum import compute_momentum, momentum_table
from trending.rank import rank_trending, rank_emerging
from trending import render
from trending.pipeline import commodity_lists

⏱ 0.0 s


## 1. Configuration

Pick the commodities to track. Each one has a Knowledge Graph query and a few search phrases (`texts`) in
`cfg.COMMODITIES`; you can add your own (see the last section).

* **API key:** `BIGDATA_API_KEY` is required.
* **LLM:** OpenAI GPT 5.6 (`OPENAI_API_KEY`). Set `cfg.USE_LLM = False` to run without any AI model.

> ⏱ **Typical time:** under 1 s

In [25]:
# 1. Configuration
COMMODITIES = ["Gold", "Silver", "Copper", "Crude Oil"]   # also available: "Natural Gas", "Platinum", "Lithium"

cfg.WINDOW = "1D"          # "1D" = last 24h vs prior 7 days  |  "1W" = last 7 days vs prior 28 days
cfg.USE_LLM = True
cfg.LLM_MODEL = "gpt-5.6"
cfg.GATE_KEEP = ["market_move"]   # add "company_specific" to also keep miner/producer company news

# Emerging list: commodity news is mostly neutral price reporting, which dilutes the *average* sentiment of a
# story. "max_abs" uses the strongest single passage instead, so sharp but quiet events stay visible.
cfg.SENTIMENT_MODE = "max_abs"
cfg.SENTIMENT_EXTREME = 0.5

client, llm, win = start("commodities")
selection = {name: cfg.COMMODITIES[name] for name in COMMODITIES}

Mode: LIVE · window 1D: 2026-10-13 21:10 → 2026-10-14 21:10 UTC
LLM: LLM(provider='openai', model='gpt-5.6')
⏱ 0.0 s


## 2. Resolve commodities (Knowledge Graph)

Each commodity is looked up among Knowledge Graph concepts of type *Commodity*. If one cannot be found, it runs in
**keyword mode**: articles are found by its search phrases instead of an entity filter, and momentum is measured the
same way. The `mode` column shows which was used.

> ⏱ **Typical time:** under 1 s (a few seconds for names not seen before)

In [26]:
# 2. Resolve commodities
entities = resolve_all(client, commodities=selection, as_of=win.as_of)
entity_table(entities)

   'Gold': 1 KG commodity candidates
   'Silver': 1 KG commodity candidates
   'Copper': 1 KG commodity candidates
   'Crude Oil': 1 KG commodity candidates


,input,label,entity_id,name,kg_type,kg_category,mode,how
0,gold,Gold,9768FE,Gold,Commodity,concepts,entity,KG concepts (Commodity)
1,silver,Silver,F96E34,Silver,Commodity,concepts,entity,KG concepts (Commodity)
2,copper,Copper,A4090F,Copper,Commodity,concepts,entity,KG concepts (Commodity)
3,crude oil,Crude Oil,FD9CFE,Crude Oil,Commodity,concepts,entity,KG concepts (Commodity)


⏱ 9.0 s


## 3. Momentum: which commodities are unusually loud right now?

`spike` = documents in the current window ÷ the commodity's normal volume for a window of the same length.
Above 1 means more coverage than usual. `trend` shows daily document counts for the last 8 days.

> ⏱ **Typical time:** about 2 s

In [27]:
# 3. Momentum
def show_momentum(df):
    """Momentum table with a bar on the spike column (no extra dependencies)."""
    return (df.style
              .format({"spike": "{:.2f}×", "baseline_per_period": "{:,.0f}", "current_docs": "{:,}",
                       "sentiment": "{:+.2f}"}, na_rep="–")
              .bar(subset=["spike"], color="#f0a35e", vmin=0)
              .hide(axis="index"))

momentum = compute_momentum(client, entities, win)
show_momentum(momentum_table(momentum, entities))

entity,name,spike,current_docs,baseline_per_period,distinct_sources,sentiment,trend,mode
Copper,Copper,2.27×,566,249,168,-0.00,▄▆▅▁▁▄▅█,entity
Silver,Silver,1.87×,690,368,224,+0.03,▅▆▆▁▁▄▆█,entity
Gold,Gold,1.42×,"1,990","1,402",486,-0.00,█▇▇▁▁▅▇▇,entity
Crude Oil,Crude Oil,1.39×,"2,289","1,650",727,-0.32,▅█▅▁▁▆▆▆,entity


⏱ 2.0 s


## 4. Relevance gate, story grouping, enrichment and ranking

Keyword and entity searches for commodities also return company promotions ("Can X Gold strengthen its exploration
story?") and unrelated uses of the word. The **relevance gate** labels every article as `market_move`,
`company_specific` or `irrelevant` and keeps `market_move` by default. The table below shows how many articles fell
into each class. Then articles are grouped into stories, the leading stories are enriched (direction is required
here), and everything is scored. This is the longest step: most of it is the AI model.

> ⏱ **Typical time:** about 1-2 min

In [28]:
# 4. Relevance gate, grouping, enrichment and ranking
stories, articles, gate_stats = build_stories(client, llm, entities, win, momentum, gate=True,
                                              per_group=cfg.TOP_N_PER_GROUP, commodity=True)
keys = [e.key for e in entities]
combined, by_commodity, emerging_by_commodity = commodity_lists(stories, keys, win, use_llm=llm is not None)

display(pd.DataFrame(gate_stats).T.fillna(0).astype(int).rename_axis("relevance gate"))

cols = ["rank", "title", "entities", "direction", "driver", "articles", "sources", "spike", "materiality",
        "avg_sentiment", "trend_score"]
trending_df = render.stories_frame(combined, "trending")
print("Top trending across commodities")
trending_df[cols]

Step 3  fetched 760 unique articles: Gold 249, Silver 158, Copper 158, Crude Oil 195  (5s)
        relevance gate kept 729 articles (market_move)  (6s)
Step 4  grouped into 528 stories, 77 covered by ≥2 sources  (7s)
Step 5  enriched 61 stories (title, summary, direction, driver, materiality)  (7s)
Step 6  scored all stories


,market_move,company_specific
relevance gate,,
Gold,241,8
Silver,141,17
Copper,152,6
Crude Oil,195,0


Top trending across commodities


,rank,title,entities,direction,driver,articles,sources,spike,materiality,avg_sentiment,trend_score
0,1,Oil prices rise as Iran deploys new tactics to target ships transiting Hormuz,Crude Oil,up,,12,9,1.39,None,-0.425,1.5188
1,2,Gold Extends Losses Ahead Of US Inflation Data,Gold,down,,10,10,1.42,None,-0.115,1.4472
2,3,HudBay Minerals Inc. - Silver Near $61 Puts Canada's Northern Polymetallic Belts Back ...,Silver,mixed,,7,7,1.87,None,0.116,1.3947
3,4,"CITIC Securities: Gold prices may rebound from the bottom, and the gold sector is expe...",Gold,up,,10,8,1.42,None,0.343,1.3874
4,5,"Oil prices rise as traders weigh Saudi supply risk, Russian diesel flow",Crude Oil,up,,8,8,1.39,None,-0.237,1.3303
5,6,Copper Demand Is Forecast to Rise 50% by 2040. Mined Supply Is Expected to Peak Around...,Copper,up,,6,5,2.27,None,0.004,1.3000
6,7,Energy agency lowers world oil demand forecast for the year,Crude Oil,mixed,,7,7,1.39,None,-0.334,1.2900
7,8,Idaho Copper Announces Results of Updated Preliminary Economic Assessment for the CuMo...,Copper,mixed,,5,5,2.27,None,0.075,1.2523
8,9,Spot silver falls below $56/oz,Silver,down,,6,4,1.87,None,-0.348,1.2084
9,10,Aurubis raises European copper premium for 2027 to a record $347 per tonne,Copper,mixed,,4,4,2.27,None,0.019,1.1830


⏱ 7.5 s


### Trending stories per commodity

Each commodity's top stories, with the direction the story implies for the commodity's price.

> ⏱ **Typical time:** under 1 s

In [29]:
# 4b. Trending stories per commodity
for key, ss in by_commodity.items():
    m = momentum.get(key)
    print(f"\n{key}  ·  {m.spike:.1f}× usual coverage" if m else f"\n{key}")
    for i, s in enumerate(ss, 1):
        arrow = {"up": "▲", "down": "▼", "mixed": "◆"}.get(s.direction, "·")
        print(f"  {i}. {arrow} {s.direction:<5} {s.title}  [{s.articles} articles · {s.sources} sources · {s.driver}]")


Gold  ·  1.4× usual coverage
  1. ▼ down  Gold Extends Losses Ahead Of US Inflation Data  [10 articles · 10 sources · ]
  2. ▲ up    CITIC Securities: Gold prices may rebound from the bottom, and the gold sector is expected to lead the way.  [10 articles · 8 sources · ]
  3. ▼ down  Gold And Silver Tumble As Inflation Data Boosts Rate-Hike Expectations  [4 articles · 3 sources · ]
  4. ▼ down  Gold drops over 1% as hot US inflation data boosts Fed hike bets  [4 articles · 4 sources · ]
  5. ▼ down  [Global Finance] Inflation Fears Weigh on Precious Metals, New York Gold Prices Slightly Down, Silver Prices Drop Nearly 2%  [3 articles · 3 sources · ]

Silver  ·  1.9× usual coverage
  1. ◆ mixed HudBay Minerals Inc. - Silver Near $61 Puts Canada's Northern Polymetallic Belts Back in Focus  [7 articles · 7 sources · ]
  2. ▼ down  Spot silver falls below $56/oz  [6 articles · 4 sources · ]
  3. ▼ down  Spot silver fell 3% intraday.  [5 articles · 4 sources · ]
  4. ▼ down  Gold And Silver

## 5. Emerging signals per commodity: strong sentiment, limited coverage so far

Stories that are **not** in Trending but carry strongly positive or negative Bigdata.com sentiment from credible
outlets (a single source is allowed only for top-tier outlets, rank 1-2), have materiality 4-5, and point in a clear
direction. For example, a sharp negative silver supply story covered by one tier-1 outlet.

> ⏱ **Typical time:** under 1 s

In [30]:
# 5. Emerging signals per commodity
emerging = [s for k in keys for s in emerging_by_commodity[k]]
emerging = list({s.sid: s for s in emerging}.values())          # a story about two commodities counts once
emerging_df = render.stories_frame(emerging, "emerging")
if len(emerging_df):
    display(emerging_df[["title", "entities", "direction", "sentiment_signal", "sources", "best_source_rank",
                         "representative_source", "driver", "materiality", "emerging_score"]])
else:
    print("No Emerging stories in this window. Try cfg.SENTIMENT_EXTREME = 0.35 (see README, Tuning).")

No Emerging stories in this window. Try cfg.SENTIMENT_EXTREME = 0.35 (see README, Tuning).
⏱ 0.0 s


## 6. The report

Grouped by commodity: the header shows the momentum and the net direction of the commodity's top stories
(e.g. "Gold ↑ 2.9× · mostly up"); each commodity lists its Trending stories, then its Emerging items. A combined
"Top trending across commodities" list sits at the top. Saved to `outputs/` as HTML and CSV.

> ⏱ **Typical time:** under 1 s

In [31]:
# 6. Build and save the report
ctx = render.ReportContext(win=win, momentum=momentum, show_direction=True, names={e.key: e.name for e in entities})
TITLE = "Trending commodity news"
gate_note = (f"Relevance gate kept {sum(len(v) for v in articles.values()):,} market-moving articles "
             f"({', '.join(cfg.GATE_KEEP)}); company promotions and unrelated articles were removed.")
report = render.commodities_report(title=TITLE, scope=", ".join(COMMODITIES), ctx=ctx, combined=combined,
                                   groups=by_commodity, emerging_by_group=emerging_by_commodity,
                                   llm_label=llm_label(llm), gate_note=gate_note)
paths = render.save_outputs(report, TITLE, {"trending": trending_df, "emerging": emerging_df},
                            "commodities", day=win.as_of)
for kind, p in paths.items():
    print(f"saved {kind:<9} → {os.path.relpath(p, ROOT)}")
display(HTML(report))

saved html      → outputs/commodities_trending_2026-10-14.html
saved trending  → outputs/commodities_2026-10-14_trending.csv
saved emerging  → outputs/commodities_2026-10-14_emerging.csv


⏱ 0.0 s


In [32]:
# Record usage and prune the local response cache
finish(client, llm)

Bigdata.com query units used this session: 143.2, LLM calls 0 (0 s summed over parallel calls, 0 from cache)
Fixtures saved under fixtures/commodities/ (removed 0 stale files).
⏱ 0.0 s


## Run time

Every code cell above ends with `⏱`: how long it took in this run. The table adds them up per step. Live runs are
dominated by the AI steps (grouping and enrichment) and by Bigdata.com calls.

In [33]:
# Run time: how long each step took in this run
cell_timer.summary()

,step,time,share,seconds
0,Setup,0.0 s,0%,0.0
1,1. Configuration,0.0 s,0%,0.0
2,2. Resolve commodities,9.0 s,49%,9.0
3,3. Momentum,2.0 s,11%,2.0
4,"4. Relevance gate, grouping, enrichment and ranking",7.5 s,41%,7.5
5,4b. Trending stories per commodity,0.0 s,0%,0.0
6,5. Emerging signals per commodity,0.0 s,0%,0.0
7,6. Build and save the report,0.0 s,0%,0.0
8,Record usage and prune the local response cache,0.0 s,0%,0.0
9,Total,18.5 s,100%,18.5


## 7. How to adapt

| To… | Change |
|---|---|
| Track other commodities | Add them to `COMMODITIES` above; define new ones in `cfg.COMMODITIES`, e.g. `cfg.COMMODITIES["Iron Ore"] = {"kg_query": "iron ore", "texts": ["iron ore price", "iron ore China steel demand"]}` |
| Change what counts as relevant | `cfg.GATE_KEEP = ["market_move", "company_specific"]` to keep company news too |
| Steer what is retrieved | edit each commodity's `texts` (search phrases, max 5) |
| Look at the past week | `cfg.WINDOW = "1W"` |
| Make Emerging stricter / looser | `cfg.SENTIMENT_EXTREME` (e.g. 0.35 for more items), `cfg.SENTIMENT_MODE` (`"mean"` or `"max_abs"`), `cfg.EMERGING_MIN_MATERIALITY` |
| Show more stories per commodity | `cfg.TOP_N_PER_GROUP`, `cfg.COMBINED_TOP_N` |
| Change ranking | `cfg.WEIGHTS`, `cfg.TRENDING_MIN_SOURCES` |
| Blend in your own engagement data | `external_engagement_score(story)` in `trending/rank.py` |

See `README.md` for the full list of settings and the limitations.